# The formula and its graph, side by side

The editor is one widget; a plot is another.  `on_change` is the wire between
them: every edit committed in the editor — a replacement, a `diff`, an undo —
comes back into the kernel as an expression, and the picture is drawn from
whatever came out.

None of this is part of the library.  It is a dozen lines, and the same dozen
connect the editor to any other widget.

Needs `matplotlib`, `numpy` and `ipywidgets` beside `sympy-editor[jupyter]`.

In [ ]:
import io

import ipywidgets as widgets
import numpy as np
from matplotlib.figure import Figure
from sympy import lambdify, sin, symbols

from sympy_editor import edit

x = symbols("x")

## The picture of an expression

`lambdify` turns the expression into a numpy function, and its samples are
the curve.  Two things keep this standing when it is pointed at whatever the
editor happens to hold:

* a value that is not real — `sqrt(x)` to the left of 0 — becomes a `NaN`, so
  the curve stops instead of the plot failing;
* a pole — `1/x` — would flatten everything else, so the vertical range comes
  from the bulk of the samples rather than from their largest.

The figure is built directly, without `pyplot`: no global state, no GUI
backend, nothing that minds being drawn from the callback's thread.

In [ ]:
def samples(expr, span, n=600):
    """`expr` across `span`, with a NaN wherever it is not a real number."""
    xs = np.linspace(span[0], span[1], n)
    f = lambdify(x, expr, "numpy")
    with np.errstate(all="ignore"):                 # poles and 0/0 are the plot's business
        ys = np.asarray(f(xs), dtype=complex) + np.zeros_like(xs)   # a constant broadcasts
    return xs, np.where(np.abs(ys.imag) < 1e-9, ys.real, np.nan)


def png_of(expr, span=(-6.0, 6.0)):
    """The graph of `expr` as a PNG, ready for an `ipywidgets.Image`."""
    fig = Figure(figsize=(5.4, 3.6), dpi=110, layout="constrained")
    ax = fig.add_subplot()
    unknown = sorted(expr.free_symbols - {x}, key=str)
    try:
        if unknown:
            raise ValueError("give a value to " + ", ".join(map(str, unknown))
                             + " — the sliders below do")
        xs, ys = samples(expr, span)
    except Exception as exc:
        # A formula under construction is not always a function: say so in
        # the picture's place and wait for the next edit.
        ax.text(0.5, 0.5, str(exc), ha="center", va="center", wrap=True, fontsize=9, color="#b3261e")
        ax.set_axis_off()
    else:
        ax.axhline(0, lw=0.8, color="0.7")
        ax.axvline(0, lw=0.8, color="0.7")
        ax.plot(xs, ys, lw=2)
        finite = ys[np.isfinite(ys)]
        if finite.size:
            edge = float(np.percentile(np.abs(finite), 98)) * 1.15 or 1.0
            ax.set_ylim(-edge, edge)
        ax.set_title(str(expr), fontsize=10)
    out = io.BytesIO()
    fig.savefig(out, format="png")
    return out.getvalue()

## The two widgets

`Image` holds the PNG, and assigning to `.value` replaces the picture in
place — no flicker, no output cell cleared and redrawn.  It is a traitlet
like any other, which is what makes it safe to set from the thread the editor
answers its messages on.

In [ ]:
plot = widgets.Image(format="png", layout=widgets.Layout(width="540px"))
w = edit(sin(x) / x)


@w.on_change
def redraw(expr):
    plot.value = png_of(expr)


redraw(w.expr)                       # the first picture
widgets.HBox([w, plot], layout=widgets.Layout(align_items="center"))

### What to try

* Select `sin(x)` and pick **Transform ▾ → diff**: the curve becomes the
  derivative's.
* Type over the numerator; or put the caret after `x` and type `+ 1`.
* <kbd>Ctrl</kbd>+<kbd>Z</kbd> — an undo is a change like any other, and the
  plot follows it back.
* The picture is redrawn when a step is *committed*, not while you are
  selecting: `on_change` fires once per history step.
* On a narrow screen, `widgets.VBox` instead of `HBox` puts the graph under
  the formula.

## Parameters get sliders

A formula usually has more letters than the axis has: `a*sin(b*x) + c`.  Every
free symbol besides `x` is given a slider — made when it turns up, dropped
when an edit takes it away — and the curve is drawn with those values
substituted in.  The formula in the editor stays symbolic; the numbers stay
outside it.

In [ ]:
params, rack = {}, widgets.VBox([])
curve = widgets.Image(format="png", layout=widgets.Layout(width="540px"))
w2 = edit("a*sin(b*x) + c")


def draw(*_):
    """The formula with the sliders' values in it."""
    curve.value = png_of(w2.expr.subs({s: sl.value for s, sl in params.items()}), span=(-10.0, 10.0))


@w2.on_change
def follow(expr):
    """A slider per free symbol: new ones appear, vanished ones go."""
    wanted = sorted(expr.free_symbols - {x}, key=str)
    for s in wanted:
        if s not in params:
            slider = widgets.FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.05, description=str(s))
            slider.observe(draw, "value")
            params[s] = slider
    for gone in set(params) - set(wanted):
        params.pop(gone)
    rack.children = [params[s] for s in wanted]
    draw()


follow(w2.expr)
widgets.HBox([widgets.VBox([w2, rack]), curve], layout=widgets.Layout(align_items="center"))

Edit the formula and the rack of sliders follows it: type `d` somewhere and a
`d` slider appears, delete `c` and its slider goes.  Move a slider and only
the curve moves — the editor is not touched, because the substitution happens
on the way to the plot.

## Anything else that draws

The wire is two lines — a callback that computes, a traitlet that shows — so
the same shape connects the editor to any widget: an `Output` holding
`expr.series(x)`, a table of values, a `plotly` figure.
`plot_surface.ipynb` does the last one, where the picture updates in place
with no PNG in between.